In [1]:
import import_ipynb

import os
import cv2
import mediapipe as mp
import pandas as pd
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
from scipy.signal import find_peaks

from feature_healpers import (calculate_knee_angle, calculate_torso_angle, 
                              calculate_hip_angle, calculate_body_height, 
                              calculate_hip_height, calculate_knee_forward, 
                              calculate_hip_shift)
from squat_summary import summarize_video

In [2]:
# Initialize the modern Pose Landmarker
model_path = "pose_landmarker.task"

# Ensure the model file is present
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Please download '{model_path}' using the curl command before running.")

options = vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    running_mode=vision.RunningMode.VIDEO,
    num_poses=1
)

def extract_coordinates_from_video(video_path, label, landmarker):
    """
    Reads a video file, extracts Hip, Knee, Ankle, and Shoulder coordinates 
    for each frame using the modern Tasks API, and returns a list of rows.
    """
    cap = cv2.VideoCapture(video_path)
    video_name = os.path.basename(video_path)
    data_rows = []
    frame_count = 0

    print(f"Processing: {video_name}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Convert the frame to a MediaPipe Image object
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Calculate timestamp in milliseconds
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        timestamp_ms = int((frame_count / fps) * 1000)

        # Run inference
        results = landmarker.detect_for_video(mp_image, timestamp_ms)

        # Check if person was found
        if results.pose_landmarks:
            landmarks = results.pose_landmarks[0]

            # Landmark indices
            LEFT = {
                "shoulder": 11,
                "hip": 23,
                "knee": 25,
                "ankle": 27,
            }

            RIGHT = {
                "shoulder": 12,
                "hip": 24,
                "knee": 26,
                "ankle": 28,
            }

            # Compute average visibility for each side
            left_visibility = (
                landmarks[LEFT["shoulder"]].visibility +
                landmarks[LEFT["hip"]].visibility +
                landmarks[LEFT["knee"]].visibility +
                landmarks[LEFT["ankle"]].visibility
            ) / 4

            right_visibility = (
                landmarks[RIGHT["shoulder"]].visibility +
                landmarks[RIGHT["hip"]].visibility +
                landmarks[RIGHT["knee"]].visibility +
                landmarks[RIGHT["ankle"]].visibility
            ) / 4

            # Choose the side with better visibility
            if left_visibility >= right_visibility:
                side = LEFT
                body_side = "left"
                visibility = left_visibility
            else:
                side = RIGHT
                body_side = "right"
                visibility = right_visibility

            row = {
                "video_name": video_name,
                "frame": frame_count,
                "label": label,

                "body_side": body_side,
                "visibility": visibility,

                "shoulder_x": landmarks[side["shoulder"]].x,
                "shoulder_y": landmarks[side["shoulder"]].y,

                "hip_x": landmarks[side["hip"]].x,
                "hip_y": landmarks[side["hip"]].y,

                "knee_x": landmarks[side["knee"]].x,
                "knee_y": landmarks[side["knee"]].y,

                "ankle_x": landmarks[side["ankle"]].x,
                "ankle_y": landmarks[side["ankle"]].y,
            }
            data_rows.append(row)
        
        frame_count += 1

    cap.release()
    return data_rows

def main():
    video_folder = "./videos" 
    all_extracted_data = []
    valid_extensions = (".mp4", ".avi", ".mov")

    if not os.path.exists(video_folder):
        print(f"Error: Folder '{video_folder}' not found. Please create it.")
        return

    # Use the context manager to open and automatically close the landmarker
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        for file_name in os.listdir(video_folder):
            if file_name.lower().endswith(valid_extensions):
                video_path = os.path.join(video_folder, file_name)
                label = "good" if "good" in file_name.lower() else "bad"
                
                video_data = extract_coordinates_from_video(video_path, label, landmarker)
                all_extracted_data.extend(video_data)

    # Convert and save
    if all_extracted_data:
        df = pd.DataFrame(all_extracted_data)
        output_csv = "squat_features.csv"
        df.to_csv(output_csv, index=False)
        print(f"\nSuccess! Data saved to {output_csv} (Shape: {df.shape})")
    else:
        print("No data extracted. Check if your video files are inside the './videos' folder.")

In [3]:
main()

Processing: good_form_1.mp4...

Success! Data saved to squat_features.csv (Shape: (338, 13))


In [4]:
def add_geometry_features(df):
    """
    Compute geometric features for every frame.
    """

    df["knee_angle"] = df.apply(calculate_knee_angle, axis=1)

    df["hip_angle"] = df.apply(calculate_hip_angle, axis=1)

    df["torso_angle"] = df.apply(calculate_torso_angle, axis=1)

    df["body_height"] = df.apply(calculate_body_height, axis=1)

    df["hip_height"] = df.apply(calculate_hip_height, axis=1)

    df["knee_forward"] = df.apply(calculate_knee_forward, axis=1)

    df["hip_shift"] = df.apply(calculate_hip_shift, axis=1)

    return df

In [ ]:
INPUT_CSV = "squat_features.csv"
OUTPUT_CSV = "squat_features_engineered.csv"

SMOOTHING_WINDOW = 5
PEAK_PROMINENCE = 0.05
PEAK_DISTANCE = 20


def load_data(csv_path):
    """Load and sort pose data."""
    df = pd.read_csv(csv_path)
    df = df.sort_values(["video_name", "frame"]).reset_index(drop=True)
    return df


def add_smoothed_features(df, window=SMOOTHING_WINDOW):
    """Smooth landmark coordinates using rolling mean (reduces jitter)."""

    joints = ["shoulder", "hip", "knee", "ankle"]

    for joint in joints:
        for axis in ["x", "y"]:
            col = f"{joint}_{axis}"

            df[f"{col}_smooth"] = (
                df.groupby("video_name")[col]
                .transform(
                    lambda x: x.rolling(window=window, min_periods=1).mean()
                )
            )

    return df


def add_velocity_from_smoothed(df):
    """Compute velocity using smoothed coordinates (IMPORTANT FIX)."""

    joints = ["shoulder", "hip", "knee", "ankle"]

    for joint in joints:
        for axis in ["x", "y"]:
            smooth_col = f"{joint}_{axis}_smooth"

            df[f"{joint}_velocity_{axis}"] = (
                df.groupby("video_name")[smooth_col]
                .diff()
                .fillna(0)
            )

    # Useful derived metric
    df["hip_speed"] = df["hip_velocity_y"].abs()

    return df


def add_relative_position_features(df):
    """Body structure relationships (frame-based, no change needed)."""

    df["hip_to_knee_y"] = df["hip_y"] - df["knee_y"]
    df["shoulder_to_hip_y"] = df["shoulder_y"] - df["hip_y"]

    return df


def save_data(df, output_path):
    """Save final dataset."""
    df.to_csv(output_path, index=False)
    print(f"Saved engineered features to '{output_path}'")
    print(f"Shape: {df.shape}")

def add_repetitions_column(df, prominence=PEAK_PROMINENCE, distance=PEAK_DISTANCE):
    """
    Automatically segments and numbers individual squat repetitions per video
    """
    df['rep_number'] = 0
    unique_videos = df['video_name'].unique()

    for video_name in unique_videos:
        video_indices = df[df['video_name'] == video_name].index
        video_df = df.loc[video_indices].sort_values('frame')
        
        signal = video_df['hip_y_smooth'].values
        bottom_indices, _ = find_peaks(signal, prominence=prominence, distance=distance)
        
        if len(bottom_indices) == 0:
            continue
            
        for i, bottom_idx in enumerate(bottom_indices):
            if i == 0:
                start_idx = np.argmin(signal[:bottom_idx])
            else:
                prev_bottom = bottom_indices[i-1]
                start_idx = prev_bottom + np.argmin(signal[prev_bottom:bottom_idx])
                
            if i == len(bottom_indices) - 1:
                end_idx = bottom_idx + np.argmin(signal[bottom_idx:])
            else:
                next_bottom = bottom_indices[i+1]
                end_idx = bottom_idx + np.argmin(signal[bottom_idx:next_bottom])
            
            actual_row_start = video_df.index[start_idx]
            actual_row_end = video_df.index[end_idx]
            
            df.loc[actual_row_start:actual_row_end, 'rep_number'] = i + 1

    return df


def main():
    df = load_data(INPUT_CSV)

    df = add_smoothed_features(df)
    df = add_repetitions_column(df)
    df = add_velocity_from_smoothed(df)
    df = add_relative_position_features(df)
    
    # Should be moved from here
    # df = add_geometry_features(df)

    save_data(df, OUTPUT_CSV)

    return df

In [6]:
df = main()

Saved engineered features to 'squat_features_engineered.csv'
Shape: (338, 33)


In [7]:
def build_video_dataset(frame_df, output_path="squat_dataset_final.csv"):
    """
    Converts frame-level data into video-level dataset
    and saves it as CSV.
    """

    video_rows = []

    for video_name, video_df in frame_df.groupby("video_name"):

        features = summarize_video(video_df)
        features["video_name"] = video_name

        video_rows.append(features)

    final_df = pd.DataFrame(video_rows)

    final_df.to_csv(output_path, index=False)

    print(f"Saved dataset to: {output_path}")
    print(f"Shape: {final_df.shape}")

    return final_df

In [8]:
build_video_dataset(df)

KeyError: 'hip_height'